In [41]:
import re
import json
from pathlib import Path

In [42]:
INPUT_DIR = 'dataset/figma-data/split/components'
OUTPUT_DIR = 'dataset/figma-data/cleaned/components'
EXTRACT_DOCUMENT_FIRST = True

PRUNE_INVISIBLE = True
DROP_EMPTY_CHILDREN = True

In [43]:
KEEP_BY_TYPE: dict[str, dict[str, set]] = {
    'FRAME': {
        'always': {'id', 'name', 'type'},
        'conditional': {
            'children',
            'absoluteBoundingBox',
            'layoutMode',
            'itemSpacing',
            'paddingLeft', 'paddingRight', 'paddingTop', 'paddingBottom',
            'primaryAxisAlignItems', 'counterAxisAlignItems',
            'layoutSizingHorizontal', 'layoutSizingVertical',
            'layoutWrap',
            'layoutGrow',
            'layoutAlign',
            'cornerRadius',
            'rectangleCornerRadii',
            'fills',
            'strokes',
        },
    },
    'INSTANCE': {
        'always': {'id', 'name', 'type', 'componentId'},
        'conditional': {
            'children',
            'absoluteBoundingBox',
            'componentProperties',
            'componentPropertyReferences',
            'isExposedInstance',
            'fills',
            'layoutGrow',
            'layoutAlign',
        },
    },
    'TEXT': {
        'always': {'id', 'name', 'type'},
        'conditional': {
            'absoluteBoundingBox',
            'characters',
            'componentPropertyReferences',
            'style',
        },
    },
    'RECTANGLE': {
        'always': {'id', 'name', 'type'},
        'conditional': {
            'absoluteBoundingBox',
            'cornerRadius',
            'fills',
        },
    },
    'VECTOR': {
        'always': {'id', 'name', 'type'},
        'conditional': set(),
    },
}

# Fallback for node types not explicitly configured above (e.g. SECTION).
# Includes absoluteBoundingBox so unconfigured types don't lose their
# layout footprint entirely.
DEFAULT_KEEP = {
    'always': {'id', 'name', 'type'},
    'conditional': {'children', 'absoluteBoundingBox'},
}

TEXT_STYLE_KEEP = {'fontFamily', 'fontWeight', 'fontSize'}

OMIT_WHEN_DEFAULT = {
    'layoutMode': 'NONE',
    'layoutWrap': 'NO_WRAP',
    'itemSpacing': 0,
    'cornerRadius': 0,
    'paddingLeft': 0,
    'paddingRight': 0,
    'paddingTop': 0,
    'paddingBottom': 0,
    'layoutGrow': 0,
}

print('Prune invisible nodes:', PRUNE_INVISIBLE)
print('Drop empty children:', DROP_EMPTY_CHILDREN)
print('Extract document first:', EXTRACT_DOCUMENT_FIRST)

print('Configured Node-Typs:', list(KEEP_BY_TYPE.keys()))
print('Configured Edge-Typs:', list(DEFAULT_KEEP.keys()))
print('Configured Text Styles:', list(TEXT_STYLE_KEEP))

print('Omit when default:', OMIT_WHEN_DEFAULT)

Prune invisible nodes: True
Drop empty children: True
Extract document first: True
Configured Node-Typs: ['FRAME', 'INSTANCE', 'TEXT', 'RECTANGLE', 'VECTOR']
Configured Edge-Typs: ['always', 'conditional']
Configured Text Styles: ['fontSize', 'fontWeight', 'fontFamily']
Omit when default: {'layoutMode': 'NONE', 'layoutWrap': 'NO_WRAP', 'itemSpacing': 0, 'cornerRadius': 0, 'paddingLeft': 0, 'paddingRight': 0, 'paddingTop': 0, 'paddingBottom': 0, 'layoutGrow': 0}


In [44]:
def clean_fills(fills) -> list[dict] | None:
    if not isinstance(fills, list):
        return None

    result = []

    for fill in fills:
        if not isinstance(fill, dict):
            continue

        if fill.get('type') == 'SOLID' and 'color' in fill:
            color = fill['color']

            result.append({
                'type': 'SOLID',
                'color': {k: round(v, 4) for k, v in color.items()},
            })

    return result if result else None

def clean_text_style(style) -> dict:
    if not isinstance(style, dict):
        return {}

    return {k: v for k, v in style.items() if k in TEXT_STYLE_KEEP}

def clean_component_properties(props) -> dict:
    """Flatten Figma's verbose {key: {value, type, boundVariables}} into
    {key: value}, and strip the '#<node-id>' suffix Figma appends to
    component property keys (e.g. 'Text#4271:0' -> 'Text')."""
    if not isinstance(props, dict):
        return {}

    result = {}
    for key, val in props.items():
        clean_key = re.sub(r'#.*$', '', key)
        result[clean_key] = val.get('value') if isinstance(val, dict) else val

    return result

def should_omit(key: str, value) -> bool:
    if value is None:
        return True

    if key in OMIT_WHEN_DEFAULT and value == OMIT_WHEN_DEFAULT[key]:
        return True

    return False

In [45]:
def clean_node(node, prune_invisible: bool = True, components_lookup: dict | None = None) -> dict | None:
    if not isinstance(node, dict):
        return None

    # Remove invisible nodes (if configured)
    if prune_invisible and node.get('visible') is False:
        return None

    components_lookup = components_lookup or {}
    nt = node.get('type', '_UNKNOWN')
    spec = KEEP_BY_TYPE.get(nt, DEFAULT_KEEP)
    cleaned: dict = {}

    # NOTE: internal PrimeVue sub-instances (name starts with '_') are
    # deliberately recursed into just like any other node - their content
    # can carry meaningful overrides (e.g. custom text, state) even though
    # they're Figma-internal implementation nodes. The component inventory
    # step further below still skips them for the *component catalog*,
    # since that step counts top-level component usage, not tree content.

    # 1. Always-Keys in strict order
    for key in ('type', 'id', 'name', 'componentId'):
        if key not in spec['always'] or key not in node:
            continue

        value = node[key]

        # Resolve componentId to the actual component name via the
        # components lookup table (from the raw payload's top-level
        # 'components' dict) - a raw ID is meaningless to an LLM.
        if key == 'componentId':
            comp = components_lookup.get(value)
            value = comp['name'] if comp else value

        cleaned[key] = value

    # 2. Conditional-Keys – without children (come at end)
    for key in sorted(spec['conditional']):
        if key == 'children' or key not in node:
            continue

        value = node[key]

        if should_omit(key, value):
            continue

        # Special cleaning
        if key == 'fills':
            value = clean_fills(value)

            if value is None:
                continue

        elif key == 'strokes':
            value = clean_fills(value)

            if value is None:
                continue

        elif key == 'style':
            value = clean_text_style(value)

            if not value:
                continue

        elif key == 'componentProperties':
            value = clean_component_properties(value)

            if not value:
                continue

        elif key == 'absoluteBoundingBox':
            if not isinstance(value, dict):
                continue
            value = {
                'x': round(value.get('x', 0), 1),
                'y': round(value.get('y', 0), 1),
                'width': round(value.get('width', 0), 1),
                'height': round(value.get('height', 0), 1),
            }

        cleaned[key] = value

    # 3. Children recursively, always at the end
    if 'children' in spec['conditional'] and 'children' in node:
        cleaned_children = []

        for child in node['children'] or []:

            cc = clean_node(child, prune_invisible, components_lookup)

            if cc is not None:
                cleaned_children.append(cc)

        if cleaned_children:
            cleaned['children'] = cleaned_children
        elif not DROP_EMPTY_CHILDREN:
            cleaned['children'] = []

    return cleaned

In [46]:
components_lookup = {
    'COMP_KNOWN': {'name': 'Button/Size=Large, Severity=Primary'},
}

test_node = {
    'id': '1:1', 'name': 'test', 'type': 'FRAME',
    'scrollBehavior': 'SCROLLS', 'blendMode': 'PASS_THROUGH',
    'layoutMode': 'VERTICAL', 'itemSpacing': 12,
    'paddingLeft': 0, 'paddingRight': 16,
    'layoutGrow': 0,  # default -> should be omitted
    'strokes': [
        {'type': 'GRADIENT_LINEAR', 'gradientStops': []},  # should be filtered out
        {'type': 'SOLID', 'color': {'r': 0.1, 'g': 0.2, 'b': 0.3, 'a': 1}},  # should survive
    ],
    'effects': [], 'interactions': [],
    'children': [
        # 1. invisible node -> pruned entirely
        {'id': '1:2', 'name': 'hidden', 'type': 'TEXT', 'visible': False, 'characters': 'gone'},

        # 2. visible TEXT with style keys beyond TEXT_STYLE_KEEP -> filtered down
        {
            'id': '1:3', 'name': 'visible', 'type': 'TEXT', 'characters': 'kept',
            'style': {
                'fontFamily': 'Inter', 'fontWeight': 400, 'fontSize': 16,
                'textAlignHorizontal': 'LEFT', 'lineHeightPx': 20,  # should be dropped
            },
        },

        # 3. known INSTANCE: componentId resolvable, componentProperties with
        #    '#nodeid' suffix + nested {value,type,boundVariables}, gradient-only
        #    fill (fully dropped), layoutGrow=1 (kept, non-default)
        {
            'id': '1:4', 'name': 'button', 'type': 'INSTANCE', 'componentId': 'COMP_KNOWN',
            'layoutGrow': 1,
            'fills': [{'type': 'GRADIENT_LINEAR', 'gradientStops': []}],
            'componentProperties': {
                'Label#1729:0': {'value': 'Save', 'type': 'TEXT', 'boundVariables': {}},
                'Severity': {'value': 'Primary', 'type': 'VARIANT', 'boundVariables': {}},
            },
            'children': [
                {'id': '1:5', 'name': 'label', 'type': 'TEXT', 'characters': 'Save'},
            ],
        },

        # 4. unknown INSTANCE: componentId NOT in lookup -> stays as raw id,
        #    empty componentProperties -> omitted, empty children -> 'children' dropped
        {
            'id': '1:6', 'name': 'icon-only-button', 'type': 'INSTANCE',
            'componentId': 'COMP_UNKNOWN_ID',
            'componentProperties': {},
            'children': [],
        },

        # 5. internal PrimeVue sub-instance ('_' prefix) - content must be KEPT
        #    (per requirement: overrides inside sub-instances matter)
        {
            'id': '1:7', 'name': '_inputtext-content', 'type': 'INSTANCE',
            'componentId': 'COMP_UNKNOWN_ID',
            'children': [
                {'id': '1:8', 'name': 'Placeholder', 'type': 'TEXT', 'characters': 'Custom Text'},
            ],
        },

        # 6. RECTANGLE: default cornerRadius (0) omitted, SOLID fill kept
        {
            'id': '1:9', 'name': 'rect', 'type': 'RECTANGLE',
            'cornerRadius': 0,
            'fills': [{'type': 'SOLID', 'color': {'r': 1, 'g': 1, 'b': 1, 'a': 1}}],
            'absoluteBoundingBox': {'x': 10.123, 'y': -5.678, 'width': 100.0, 'height': 40.0},
        },

        # 7. VECTOR: conditional set is empty -> only id/name/type survive,
        #    absoluteBoundingBox must NOT appear even though present on raw node
        {
            'id': '1:10', 'name': 'icon', 'type': 'VECTOR',
            'absoluteBoundingBox': {'x': 0, 'y': 0, 'width': 16, 'height': 16},
        },

        # 8. unconfigured type (not in KEEP_BY_TYPE) -> falls back to DEFAULT_KEEP,
        #    which now includes absoluteBoundingBox + children
        {
            'id': '1:11', 'name': 'group', 'type': 'SECTION',
            'absoluteBoundingBox': {'x': 0, 'y': 0, 'width': 200, 'height': 200},
            'children': [
                {'id': '1:12', 'name': 'nested', 'type': 'TEXT', 'characters': 'inside section'},
            ],
        },
    ],
}

cleaned_test_node = clean_node(test_node, components_lookup=components_lookup)

print('Pre Cleaning :', json.dumps(test_node, indent=2))
print('Post Cleaning:', json.dumps(cleaned_test_node, indent=2))
print(f'Reduction percentage: {100 * (1 - len(json.dumps(cleaned_test_node)) / len(json.dumps(test_node))):.2f}%')


def find_by_id(node, target_id):
    if not isinstance(node, dict):
        return None
    if node.get('id') == target_id:
        return node
    for child in node.get('children', []) or []:
        found = find_by_id(child, target_id)
        if found is not None:
            return found
    return None


def check(label: str, condition: bool):
    print(f"{'OK  ' if condition else 'FAIL'} - {label}")


print('\nSpecial-case checks:')

check('root: default layoutGrow (0) omitted', 'layoutGrow' not in cleaned_test_node)
check(
    'root: strokes filtered down to SOLID only',
    cleaned_test_node.get('strokes') == [{'type': 'SOLID', 'color': {'r': 0.1, 'g': 0.2, 'b': 0.3, 'a': 1}}],
)

check('invisible node pruned', find_by_id(cleaned_test_node, '1:2') is None)

visible_text = find_by_id(cleaned_test_node, '1:3')
check(
    'TEXT style reduced to TEXT_STYLE_KEEP only',
    visible_text is not None and set(visible_text.get('style', {}).keys()) == {'fontFamily', 'fontWeight', 'fontSize'},
)

known_instance = find_by_id(cleaned_test_node, '1:4')
check(
    'componentId resolved via components_lookup',
    known_instance is not None and known_instance.get('componentId') == 'Button/Size=Large, Severity=Primary',
)
check(
    'componentProperties flattened + "#nodeid" suffix stripped',
    known_instance is not None and known_instance.get('componentProperties') == {'Label': 'Save', 'Severity': 'Primary'},
)
check('gradient-only fill dropped entirely', known_instance is not None and 'fills' not in known_instance)
check('non-default layoutGrow (1) kept', known_instance is not None and known_instance.get('layoutGrow') == 1)
check('known instance children still recursed', known_instance is not None and 'children' in known_instance)

unknown_instance = find_by_id(cleaned_test_node, '1:6')
check(
    'unresolved componentId kept as raw id',
    unknown_instance is not None and unknown_instance.get('componentId') == 'COMP_UNKNOWN_ID',
)
check('empty componentProperties omitted', unknown_instance is not None and 'componentProperties' not in unknown_instance)
check('empty children list dropped (DROP_EMPTY_CHILDREN)', unknown_instance is not None and 'children' not in unknown_instance)

internal_sub = find_by_id(cleaned_test_node, '1:7')
check(
    '"_"-prefixed sub-instance content is KEPT (not skipped)',
    internal_sub is not None and find_by_id(internal_sub, '1:8') is not None
    and find_by_id(internal_sub, '1:8').get('characters') == 'Custom Text',
)

rect = find_by_id(cleaned_test_node, '1:9')
check('RECTANGLE: default cornerRadius (0) omitted', rect is not None and 'cornerRadius' not in rect)
check('RECTANGLE: SOLID fill kept', rect is not None and rect.get('fills') is not None)
check(
    'absoluteBoundingBox rounded to 1 decimal',
    rect is not None and rect.get('absoluteBoundingBox') == {'x': 10.1, 'y': -5.7, 'width': 100.0, 'height': 40.0},
)

vector = find_by_id(cleaned_test_node, '1:10')
check('VECTOR: absoluteBoundingBox dropped (empty conditional set)', vector is not None and 'absoluteBoundingBox' not in vector)

section = find_by_id(cleaned_test_node, '1:11')
check('unconfigured type falls back to DEFAULT_KEEP (absoluteBoundingBox kept)', section is not None and 'absoluteBoundingBox' in section)
check('unconfigured type still recurses into children', section is not None and find_by_id(section, '1:12') is not None)

Pre Cleaning : {
  "id": "1:1",
  "name": "test",
  "type": "FRAME",
  "scrollBehavior": "SCROLLS",
  "blendMode": "PASS_THROUGH",
  "layoutMode": "VERTICAL",
  "itemSpacing": 12,
  "paddingLeft": 0,
  "paddingRight": 16,
  "layoutGrow": 0,
  "strokes": [
    {
      "type": "GRADIENT_LINEAR",
      "gradientStops": []
    },
    {
      "type": "SOLID",
      "color": {
        "r": 0.1,
        "g": 0.2,
        "b": 0.3,
        "a": 1
      }
    }
  ],
  "effects": [],
  "interactions": [],
  "children": [
    {
      "id": "1:2",
      "name": "hidden",
      "type": "TEXT",
      "visible": false,
      "characters": "gone"
    },
    {
      "id": "1:3",
      "name": "visible",
      "type": "TEXT",
      "characters": "kept",
      "style": {
        "fontFamily": "Inter",
        "fontWeight": 400,
        "fontSize": 16,
        "textAlignHorizontal": "LEFT",
        "lineHeightPx": 20
      }
    },
    {
      "id": "1:4",
      "name": "button",
      "type": "INSTANCE",

In [47]:
def count_keys(obj) -> int:
    if isinstance(obj, dict):
        return len(obj) + sum(count_keys(v) for v in obj.values())
    elif isinstance(obj, list):
        return sum(count_keys(item) for item in obj)

    return 0


def extract_document_node(data: dict) -> tuple[dict, dict]:
    """Return (document, components_lookup) from raw Figma payload.

    components_lookup maps componentId -> {'name': ..., ...} (the raw
    payload's top-level 'components' dict) so INSTANCE nodes can be
    resolved to a readable component name during cleaning instead of
    keeping a meaningless ID. Falls back to (data, {}) if the expected
    shape isn't found."""
    if not EXTRACT_DOCUMENT_FIRST or not isinstance(data, dict):
        return data, {}

    nodes = data.get('nodes')
    if not isinstance(nodes, dict) or not nodes:
        return data, {}

    # Python dict keeps insertion order: take the first node only.
    first_wrapper = next(iter(nodes.values()), None)
    if isinstance(first_wrapper, dict):
        first_document = first_wrapper.get('document')
        components = first_wrapper.get('components', {})
        if isinstance(first_document, dict):
            return first_document, (components if isinstance(components, dict) else {})

    return data, {}


INPUT_DIR_PATH = Path(INPUT_DIR)
OUTPUT_DIR_PATH = Path(OUTPUT_DIR)

input_files = sorted(INPUT_DIR_PATH.rglob('*.json'))
print(f'Found {len(input_files)} input files in {INPUT_DIR}')

results = []

for input_file in input_files:
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    data_for_cleaning, components_lookup = extract_document_node(data)
    cleaned_data = clean_node(data_for_cleaning, prune_invisible=PRUNE_INVISIBLE, components_lookup=components_lookup)

    output_file = OUTPUT_DIR_PATH / input_file.relative_to(INPUT_DIR_PATH)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(cleaned_data, f, indent=4, ensure_ascii=False)

    size_before = len(json.dumps(data))
    size_after = len(json.dumps(cleaned_data))

    attributes_before = count_keys(data)
    attributes_after = count_keys(cleaned_data)

    relative_name = str(input_file.relative_to(INPUT_DIR_PATH))
    results.append((relative_name, size_before, size_after, attributes_before, attributes_after, cleaned_data.get('name', '-')))

print('Cleaning Results:')
print(f'\t{"File":<32} {"Before":>10} {"After":>10} {"Reduction":>10} {"Name":<30}')

total_reduction = 0
attributes_before = 0
attributes_after = 0

for filename, size_before, size_after, attrs_before, attrs_after, name in results:
    reduction = 100 * (1 - size_after / size_before) if size_before > 0 else 0
    total_reduction += reduction
    attributes_before += attrs_before
    attributes_after += attrs_after

    print(f'\t{filename:<32} {size_before:>10} {size_after:>10} {reduction:>9.2f}% {attrs_before:>6} -> {attrs_after:<6} {name:<30}')

print(f'Average Reduction: {(total_reduction / len(results)):.2f}')
print(f'Average Attributes Before: {(attributes_before / len(results)):.2f}')
print(f'Average Attributes After: {(attributes_after / len(results)):.2f}')

cleaning_report = {
    'summary': {
        'total_files': len(results),
        'average_reduction_percent': round(total_reduction / len(results), 2),
        'average_attributes_before': round(attributes_before / len(results), 2),
        'average_attributes_after': round(attributes_after / len(results), 2),
    },
    'files': [
        {
            'filename': filename,
            'size_before': size_before,
            'size_after': size_after,
            'reduction_percent': round(100 * (1 - size_after / size_before), 2) if size_before > 0 else 0,
            'attributes_before': attrs_before,
            'attributes_after': attrs_after,
            'name': name,
        }
        for filename, size_before, size_after, attrs_before, attrs_after, name in results
    ],
}

report_path = Path(OUTPUT_DIR) / 'cleaning_report.json'

with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(cleaning_report, f, indent=4, ensure_ascii=False)

print(f'\nReport saved to: {report_path}')

Found 30 input files in dataset/figma-data/split/components
Cleaning Results:
	File                                 Before      After  Reduction Name                          
	hard\1.json                          134385      17777     86.77%   5503 -> 800    1 [Bearbeitungs-Dialog]       
	hard\10.json                         257405      47994     81.35%  11181 -> 2280   10 [Admin-Panel-Liste]        
	hard\2.json                          168533      27358     83.77%   7182 -> 1365   2 [Nutzer-Tabelle]            
	hard\3.json                          426292      70154     83.54%  18200 -> 3214   3 [Filter-Leiste]             
	hard\4.json                          184695      34308     81.42%   7975 -> 1609   4 [Aktions-Tabelle]           
	hard\5.json                          188210      14250     92.43%   7916 -> 690    5 [Step-Formular-Dialog]      
	hard\6.json                           80072      11716     85.37%   3196 -> 515    6 [Select] - !Ändern!         
	hard\7.json       

In [48]:
from collections import defaultdict

component_properties: dict[str, set[str]] = defaultdict(set)
component_count: dict[str, int] = defaultdict(int)

KNOWN_PV_FRAMES = {
    'card', 'dialog', 'tabs', 'datatable',
    'select', 'popover', 'breadcrumb', 'accordion',
}

def walk_for_inventory(node, inside_instance: bool = False):
    if not isinstance(node, dict):
        return

    t    = node.get('type', '')
    name = node.get('name', '')
    norm = re.sub(r'[\s\-_]+', '', name).lower()

    if t == 'INSTANCE':
        # Ignore internal sub-instances (_-prefix)
        if name.startswith('_'):
            return

        # Do not drill down into children of INSTANCE nodes —
        # they only contain Figma-internal implementation details
        if inside_instance:
            return

        component_count[name] += 1
        for prop_key in (node.get('componentProperties') or {}).keys():
            component_properties[name].add(prop_key)

        # Recursively scan children, but mark them as “inside_instance”
        for child in node.get('children', []) or []:
            walk_for_inventory(child, inside_instance=True)

        return  # No further relegation at this level

    elif t == 'FRAME' and not inside_instance:
         # Identifying FRAME-based PrimeVue compound components
        if not name.startswith('_') and norm in KNOWN_PV_FRAMES:
            component_count[name] += 1
            # FRAMEs do not have componentProperties — Slot structure as metadata
            component_properties[name].add('__frame_based__')

     # Continue into children (only outside of instances)
    if not inside_instance:
        for child in node.get('children', []) or []:
            walk_for_inventory(child, inside_instance=False)

for relative_name, *_ in results:
    output_path = OUTPUT_DIR_PATH / relative_name
    with open(output_path, 'r', encoding='utf-8') as f:
        walk_for_inventory(json.load(f))

inventory = []
for name in sorted(component_properties.keys()):
    inventory.append({
        'component_name': name,
        'instances_total': component_count[name],
        'property_count': len(component_properties[name]),
        'properties': ', '.join(sorted(component_properties[name])),
    })

print('\nComponent Inventory:')
print(f'\t{"Component Name":<30} {"Instances":>10} {"Properties":>10} {"Property Keys":<40}')

for item in inventory:
    print(f'\t{item["component_name"]:<30} {item["instances_total"]:>10} {item["property_count"]:>10} {item["properties"]:<40}')

inventory_path = Path(OUTPUT_DIR) / 'component_inventory.json'

with open(inventory_path, 'w', encoding='utf-8') as f:
    json.dump(inventory, f, indent=4, ensure_ascii=False)

print(f'Component inventory saved to: {inventory_path}')


Component Inventory:
	Component Name                  Instances Properties Property Keys                           
	accordion                               1          1 __frame_based__                         
	avatar                                 15          5 Circle, Show Badge, Size, Text, Type    
	breadcrumb                              2          1 __frame_based__                         
	button                                 49         15 Disabled, Icon, Icon Only, Left Icon, Link, Right Icon, Severity, Show Left Icon, Show Right Icon, State, Text, ⥰ Rounded, ⬆️ Raised, 🔤 Text, 🔲 Outlined
	card                                    9          1 __frame_based__                         
	checkbox                               13          8 Disabled, Filled, Focus, Hover, Label, Selected, Show Label, Size
	datatable                               3          1 __frame_based__                         
	datepicker                              3          5 Picker Type, Show Bar, Show